# EDA — matched-trip TTE datasets

One notebook for every city that follows the **gold contract** (the Harbin file
format with the Omsk file naming), described in `RESEARCH.md`:

| file | contents |
|---|---|
| `matched_trips_<city>.csv` | unnamed index, `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time` |
| `edge_list_directed_<city>.csv` | `osmid_u`, `osmid_v` — directed segment-to-segment transitions |
| `road_network_unique_osmids_<city>.geojson` | one `LineString` per segment (`osmid`, `original_osmid`, `is_duplicate`, `duplicate_index`, `length`, `highway`) |

`Coordinates` / `OSMids` / `Timestamps` are equal-length Python-literal lists, one
entry per GPS fix, and `Total_time = Timestamps[-1] - Timestamps[0]`.

**Switch the dataset in the first cell** (`DATA_DIR` + `CITY`), then *Run All*. Every
section below is written against whichever city is selected; sections whose input is
missing (no geojson, no coordinates) skip themselves with a note instead of failing.

Section 11 collects the per-city summaries each run writes to `eda_out/`, so running
the notebook once per city gives a side-by-side comparison table.

> The two files shipped in `datasets.zip` are **cropped** (`*_mini`, first 1000 trips;
> the Harbin edge list is truncated as well). Counts here describe the crop, not the
> full city — the checks that are sensitive to that are flagged as such.

## 0. Dataset switch

In [ ]:
from pathlib import Path

DATA_DIR = Path("datasets")   # directory holding the csv / geojson files
CITY     = "omsk"             # <<< switch the dataset here: "omsk" | "harbin" | ...
OUT_DIR  = Path("eda_out")    # per-city summaries + saved figures land here

# File names per city.  The naming convention we standardise on is the Omsk one
# (`matched_trips_<city>.csv`, `edge_list_directed_<city>.csv`,
# `road_network_unique_osmids_<city>.geojson`); cities that arrived with other
# names are spelled out here.  A city that is not listed falls back to the
# convention, so a freshly generated dataset needs no entry at all.
DATASETS = {
    "omsk": {
        "trips":   "matched_trips_omsk_mini.csv",
        "edges":   "edge_list_directed_omsk.csv",
        "network": "road_network_unique_osmids_omsk.geojson",
        "note":    "cropped to the first 1000 trips; no geojson in datasets.zip",
    },
    "harbin": {
        "trips":   "matched_trips_harbin_mini.csv",
        "edges":   "Harbin_edge_list.csv",          # off-convention name
        "network": "road_network_unique_osmids_harbin.geojson",
        "note":    "cropped to the first 1000 trips; edge list truncated too",
    },
    "quebec": {                                     # produced by prepare_dataset_quebec.ipynb
        "trips":   "matched_trips_quebec.csv",
        "edges":   "edge_list_directed_quebec.csv",
        "network": None,                            # no geometry is published
        "note":    "Coordinates are (None, None) placeholders",
    },
}

In [ ]:
import ast, json, math, zipfile, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)

# One categorical order, used in fixed order and never cycled.  Most panels below
# are single-series, so they take slot 1 and need no legend.
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e3e2df"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

def hist(ax, values, bins=40, log=False, color=C1, title="", xlabel=""):
    """Single-series histogram with recessive furniture and a median marker."""
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], dtype=float)
    if log:
        v = v[v > 0]
        bins = np.logspace(np.log10(v.min()), np.log10(v.max()), bins)
    ax.hist(v, bins=bins, color=color, edgecolor="white", linewidth=0.4)
    if log:
        ax.set_xscale("log")
    med = float(np.median(v))
    ax.axvline(med, color=INK, linewidth=1, linestyle="--")
    ax.annotate(f"median {med:,.4g}", (med, ax.get_ylim()[1] * 0.92), color=INK,
                fontsize=8, ha="left", va="top", xytext=(4, 0), textcoords="offset points")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("trips")
    ax.set_axisbelow(True)
    return ax

def haversine_km(lon1, lat1, lon2, lat2):
    """Great-circle distance in km, vectorised over numpy arrays."""
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

def runs(seq):
    """Collapse consecutive repeats: ['a','a','b','a'] -> ['a','b','a']."""
    return [x for i, x in enumerate(seq) if i == 0 or x != seq[i - 1]]

def pct(x, n):
    return f"{x:,} ({0.0 if not n else 100.0 * x / n:.1f}%)"

In [ ]:
# datasets.zip ships the csv files; extract on first run so the notebook works from
# a clean checkout.
if not DATA_DIR.exists() and Path("datasets.zip").exists():
    with zipfile.ZipFile("datasets.zip") as z:
        z.extractall(DATA_DIR)
    print(f"extracted datasets.zip -> {DATA_DIR}/")

cfg = DATASETS.get(CITY, {})
NOTE = cfg.get("note", "")

def resolve(kind, default_name):
    """Configured name first, then the naming convention, then a loose glob."""
    name = cfg.get(kind, default_name)
    if name is None:
        return None
    p = DATA_DIR / name
    if p.exists():
        return p
    for cand in (DATA_DIR / default_name, *sorted(DATA_DIR.glob(f"*{kind[:4]}*{CITY}*"))):
        if cand.exists():
            return cand
    return p           # keep the expected path so the message below names it

TRIPS_PATH = resolve("trips",   f"matched_trips_{CITY}.csv")
EDGES_PATH = resolve("edges",   f"edge_list_directed_{CITY}.csv")
GEOJSON_PATH = resolve("network", f"road_network_unique_osmids_{CITY}.geojson")

OUT_DIR.mkdir(exist_ok=True)
print(f"city        {CITY}"      + (f"   ({NOTE})" if NOTE else ""))
for label, p in [("trips", TRIPS_PATH), ("edge list", EDGES_PATH), ("road network", GEOJSON_PATH)]:
    if p is None:
        print(f"{label:<12} — not published for this city")
    else:
        ok = "ok " if p.exists() else "MISSING"
        size = f"{p.stat().st_size / 1e6:8.2f} MB" if p.exists() else " " * 11
        print(f"{label:<12} {ok} {size}  {p}")

assert TRIPS_PATH.exists(), f"{TRIPS_PATH} not found — check DATA_DIR / CITY / DATASETS"

## 1. The files as they sit on disk

Before pandas gets a chance to normalise anything: how big the files are, what the
header line actually says, and what one raw record looks like.

In [ ]:
def peek(path, n_chars=300):
    with open(path, encoding="utf-8") as fh:
        header = fh.readline().rstrip("\n")
        first  = fh.readline().rstrip("\n")
    with open(path, "rb") as fh:
        n_lines = sum(buf.count(b"\n") for buf in iter(lambda: fh.read(1 << 20), b""))
    print(f"{path.name}\n  {n_lines:,} lines (incl. header), {path.stat().st_size / 1e6:.2f} MB")
    print(f"  header: {header}")
    print(f"  row 1 : {first[:n_chars]}{' …' if len(first) > n_chars else ''}\n")

peek(TRIPS_PATH)
if EDGES_PATH is not None and EDGES_PATH.exists():
    peek(EDGES_PATH)

## 2. Schema check against the gold contract

The contract is exact: an unnamed index column (pandas reads it as `Unnamed: 0`),
then `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time`. Anything else is a
format break that will bite the loader downstream.

In [ ]:
raw = pd.read_csv(TRIPS_PATH)
EXPECTED = ["Unnamed: 0", "Id", "Coordinates", "OSMids", "Timestamps", "Total_time"]

print(f"{len(raw):,} rows x {raw.shape[1]} columns")
print(f"columns  {list(raw.columns)}")
print(f"contract {EXPECTED}")
print("match   ", "yes" if list(raw.columns) == EXPECTED else "NO  <-- format break")
print()
print(raw.dtypes.to_string())
print()
print("index column is a plain 0..n-1 range:",
      bool((raw[raw.columns[0]].values == np.arange(len(raw))).all()))
id0 = raw["Id"].iloc[0]
print(f"Id dtype {raw['Id'].dtype} — sample {str(id0)!r}")
print("duplicate Id values:", int(raw["Id"].duplicated().sum()))
print("null cells per column:", raw.isna().sum().to_dict())

In [ ]:
# Parse the three list columns once; everything downstream works off these.
def parse_lists(df):
    parsed, bad = {}, []
    for col in ("Coordinates", "OSMids", "Timestamps"):
        out = []
        for i, s in enumerate(df[col]):
            try:
                out.append(ast.literal_eval(s))
            except Exception as exc:                     # keep going, report at the end
                out.append(None)
                bad.append((i, col, type(exc).__name__))
        parsed[col] = out
    return parsed, bad

parsed, bad_parse = parse_lists(raw)
coords_all, osmids_all, ts_all = parsed["Coordinates"], parsed["OSMids"], parsed["Timestamps"]
print(f"rows that failed literal_eval: {len(bad_parse)}")
for b in bad_parse[:5]:
    print("  ", b)

keep = [i for i in range(len(raw)) if None not in (coords_all[i], osmids_all[i], ts_all[i])]
if len(keep) < len(raw):
    print(f"dropping {len(raw) - len(keep)} unparseable rows from the analysis")
df      = raw.iloc[keep].reset_index(drop=True)
coords  = [coords_all[i] for i in keep]
osmids  = [[str(x) for x in osmids_all[i]] for i in keep]
stamps  = [ts_all[i] for i in keep]

# The three lists must line up one-to-one with the GPS fixes, and Total_time must be
# the span of the timestamps — those two invariants are what the format is *for*.
len_ok   = np.array([len(c) == len(o) == len(t) for c, o, t in zip(coords, osmids, stamps)])
span     = np.array([t[-1] - t[0] for t in stamps])
total_ok = span == df["Total_time"].values
mono_ok  = np.array([all(b >= a for a, b in zip(t, t[1:])) for t in stamps])
strict   = np.array([all(b >  a for a, b in zip(t, t[1:])) for t in stamps])

n = len(df)
print()
print("len(Coordinates) == len(OSMids) == len(Timestamps) :", pct(int(len_ok.sum()), n))
print("Total_time == Timestamps[-1] - Timestamps[0]       :", pct(int(total_ok.sum()), n))
print("Timestamps non-decreasing                          :", pct(int(mono_ok.sum()), n))
print("Timestamps strictly increasing                     :", pct(int(strict.sum()), n))

has_coords = any(isinstance(p, (tuple, list)) and len(p) == 2 and p[0] is not None
                 for c in coords for p in c[:1])
print("coordinates present (not None placeholders)        :", has_coords)

In [ ]:
# What a record looks like once parsed.
i = 0
print(f"Id          {str(df['Id'].iloc[i])!r}")
print(f"n fixes     {len(coords[i])}")
print(f"Total_time  {df['Total_time'].iloc[i]} s")
print(f"Coordinates {coords[i][:3]} …")
print(f"OSMids      {osmids[i][:3]} …  ({len(set(osmids[i]))} distinct, {len(runs(osmids[i]))} traversals)")
print(f"Timestamps  {stamps[i][:3]} …  -> {pd.to_datetime(stamps[i][0], unit='s')} UTC")

## 3. Per-trip feature table

One row per trip: the shape of the trajectory, its duration, how far it travelled and
how fast, plus the sampling cadence. This frame is the basis for everything in
sections 4–6.

In [ ]:
rows = []
for k in range(len(df)):
    c, o, t = coords[k], osmids[k], stamps[k]
    tarr = np.asarray(t, dtype="int64")
    gaps = np.diff(tarr)
    seq  = runs(o)

    lon = np.array([p[0] if p and p[0] is not None else np.nan for p in c], dtype=float)
    lat = np.array([p[1] if p and p[1] is not None else np.nan for p in c], dtype=float)
    if np.isfinite(lon).sum() > 1:
        step_km  = haversine_km(lon[:-1], lat[:-1], lon[1:], lat[1:])
        path_km  = float(np.nansum(step_km))
        od_km    = float(haversine_km(lon[0], lat[0], lon[-1], lat[-1]))
        dup_frac = float(np.mean((lon[:-1] == lon[1:]) & (lat[:-1] == lat[1:])))
    else:
        path_km = od_km = dup_frac = np.nan

    dur_h = (tarr[-1] - tarr[0]) / 3600
    rows.append({
        "Id": df["Id"].iloc[k],
        "n_fixes": len(c),
        "n_segments": len(set(o)),
        "n_traversals": len(seq),          # consecutive repeats collapsed
        "n_transitions": max(len(seq) - 1, 0),
        "total_time_s": int(df["Total_time"].iloc[k]),
        "path_km": path_km,
        "od_km": od_km,
        "detour_ratio": path_km / od_km if od_km and od_km > 0 else np.nan,
        "mean_speed_kmh": path_km / dur_h if dur_h > 0 else np.nan,
        "gap_median_s": float(np.median(gaps)) if len(gaps) else np.nan,
        "gap_max_s": float(gaps.max()) if len(gaps) else np.nan,
        "zero_gap_frac": float(np.mean(gaps == 0)) if len(gaps) else np.nan,
        "dup_coord_frac": dup_frac,        # consecutive identical fixes
        "repeat_osmid_frac": float(np.mean([a == b for a, b in zip(o, o[1:])])) if len(o) > 1 else np.nan,
        "start_ts": int(tarr[0]),
        "end_ts": int(tarr[-1]),
    })

trips = pd.DataFrame(rows)
trips["start_dt"] = pd.to_datetime(trips["start_ts"], unit="s", utc=True)
trips["hour"]     = trips["start_dt"].dt.hour
trips["weekday"]  = trips["start_dt"].dt.day_name().str[:3]
trips["date"]     = trips["start_dt"].dt.date
trips.head()

In [ ]:
cols = ["n_fixes", "n_traversals", "total_time_s", "path_km", "od_km",
        "detour_ratio", "mean_speed_kmh", "gap_median_s", "dup_coord_frac"]
trips[cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.round(3)

## 4. Distributions

Durations, trajectory length and speed on a log x-axis — trip data is heavy-tailed and
a linear axis hides the body of the distribution in the first bin.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7))
hist(axes[0, 0], trips["total_time_s"],   log=True,  title="Trip duration",        xlabel="Total_time (s, log)")
hist(axes[0, 1], trips["n_fixes"],        log=True,  title="GPS fixes per trip",   xlabel="fixes (log)")
hist(axes[0, 2], trips["n_traversals"],   log=True,  title="Segment traversals per trip", xlabel="traversals (log)")
if trips["path_km"].notna().any():
    hist(axes[1, 0], trips["path_km"],        log=True, color=C2, title="Path length",   xlabel="km (log)")
    hist(axes[1, 1], trips["mean_speed_kmh"], bins=40,  color=C2, title="Mean speed",    xlabel="km/h")
    hist(axes[1, 2], trips["detour_ratio"].clip(upper=6), bins=40, color=C2,
         title="Detour ratio (path / straight line)", xlabel="ratio, clipped at 6")
else:
    for ax in axes[1]:
        ax.text(0.5, 0.5, "no coordinates in this dataset", ha="center", va="center",
                color=MUTED, transform=ax.transAxes); ax.set_axis_off()
fig.suptitle(f"{CITY} — trip-level distributions  (n = {len(trips):,})",
             fontsize=13, fontweight="bold", color=INK, y=1.0)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{CITY}_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# The tails are where the filtering thresholds live, so print them as numbers too.
q = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
tail = trips[["total_time_s", "n_fixes", "n_traversals", "path_km", "mean_speed_kmh"]].quantile(q).round(2)
tail.index = [f"p{p*100:g}" for p in q]
print(tail.to_string())
print()
short = int((trips.total_time_s < 60).sum())
tiny  = int((trips.n_traversals < 3).sum())
fast  = int((trips.mean_speed_kmh > 120).sum()) if trips.mean_speed_kmh.notna().any() else 0
still = int((trips.mean_speed_kmh < 2).sum())  if trips.mean_speed_kmh.notna().any() else 0
print("trips shorter than 60 s      :", pct(short, len(trips)))
print("trips with < 3 traversals    :", pct(tiny, len(trips)))
print("trips faster than 120 km/h   :", pct(fast, len(trips)))
print("trips slower than 2 km/h     :", pct(still, len(trips)))

## 5. Temporal coverage

What period the crop covers, and how the trips fall across the day and the week. A
sample that is all rush hour (or all one day) cannot support time-of-day experiments —
that is exactly the trap `RESEARCH.md` flags for Quebec.

In [ ]:
t0 = trips["start_dt"].min()
t1 = trips["start_dt"].max()                                   # last *start*
t_end = pd.to_datetime(trips["end_ts"].max(), unit="s", utc=True)   # last fix of any trip
span_starts, span_all = t1 - t0, t_end - t0
print(f"first trip starts  {t0}")
print(f"last trip starts   {t1}")
print(f"last fix recorded  {t_end}")
print(f"span of starts     {span_starts.days} d {span_starts.seconds // 3600} h "
      f"{span_starts.seconds % 3600 // 60} min")
print(f"span incl. driving {span_all.days} d {span_all.seconds // 3600} h "
      f"{span_all.seconds % 3600 // 60} min")
print(f"distinct dates     {trips['date'].nunique()}")
print(f"distinct start hours {trips['hour'].nunique()} / 24")
print()
print(trips["date"].value_counts().sort_index().to_string())
if span_starts < pd.Timedelta("1h"):
    print("\n!! every trip in this file starts within the hour — the crop is sorted by\n"
          "   start time, so it is the beginning of the feed and not a sample of the city.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))

by_hour = trips["hour"].value_counts().reindex(range(24), fill_value=0)
axes[0].bar(by_hour.index, by_hour.values, color=C1, edgecolor="white", linewidth=0.4)
axes[0].set_title("Trip starts by hour (UTC)"); axes[0].set_xlabel("hour"); axes[0].set_ylabel("trips")
axes[0].set_xticks(range(0, 24, 3)); axes[0].set_axisbelow(True)

order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow = trips["weekday"].value_counts().reindex(order, fill_value=0)
axes[1].bar(by_dow.index, by_dow.values, color=C1, edgecolor="white", linewidth=0.4)
axes[1].set_title("Trip starts by weekday"); axes[1].set_ylabel("trips"); axes[1].set_axisbelow(True)

gaps = np.concatenate([np.diff(np.asarray(t, dtype="int64")) for t in stamps])
gaps_pos = gaps[gaps > 0]
axes[2].hist(gaps_pos, bins=np.logspace(0, np.log10(max(gaps_pos.max(), 10)), 40),
             color=C3, edgecolor="white", linewidth=0.4)
axes[2].set_xscale("log")
axes[2].set_title("Sampling interval between fixes")
axes[2].set_xlabel("seconds (log), zero gaps excluded"); axes[2].set_ylabel("fixes")
axes[2].set_axisbelow(True)

fig.suptitle(f"{CITY} — temporal coverage", fontsize=13, fontweight="bold", color=INK, y=1.04)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{CITY}_temporal.png", bbox_inches="tight")
plt.show()

print(f"zero-length gaps (same second twice): {pct(int((gaps == 0).sum()), len(gaps))}")
print("interval quantiles (s):",
      {f"p{p}": float(np.percentile(gaps, p)) for p in (5, 25, 50, 75, 95, 99)})

## 6. Spatial coverage

Bounding box, where the fixes are, and where trips start and end. The point cloud is
the fastest way to spot map-matching escapees — a handful of fixes far outside the city
box usually means a bad match rather than a real trip.

In [ ]:
if not has_coords:
    print("no coordinates in this dataset — spatial section skipped")
else:
    lon_all = np.array([p[0] for c in coords for p in c if p and p[0] is not None])
    lat_all = np.array([p[1] for c in coords for p in c if p and p[1] is not None])
    print(f"{len(lon_all):,} fixes")
    print(f"lon  {lon_all.min():.5f} .. {lon_all.max():.5f}   (span {(lon_all.max()-lon_all.min()):.4f}°)")
    print(f"lat  {lat_all.min():.5f} .. {lat_all.max():.5f}   (span {(lat_all.max()-lat_all.min()):.4f}°)")
    w_km = haversine_km(lon_all.min(), lat_all.mean(), lon_all.max(), lat_all.mean())
    h_km = haversine_km(lon_all.mean(), lat_all.min(), lon_all.mean(), lat_all.max())
    print(f"bounding box ≈ {w_km:.1f} km x {h_km:.1f} km")

    # points more than 3 IQR outside the central box — candidate bad matches
    def outliers(a):
        q1, q3 = np.percentile(a, [25, 75]); iqr = q3 - q1
        return (a < q1 - 3 * iqr) | (a > q3 + 3 * iqr)
    out = outliers(lon_all) | outliers(lat_all)
    print("fixes far outside the central box:", pct(int(out.sum()), len(lon_all)))

In [ ]:
if has_coords:
    lat_mid = float(np.mean(lat_all))
    aspect  = 1 / math.cos(math.radians(lat_mid))     # keep the map roughly to scale

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
    step = max(1, len(lon_all) // 60000)
    axes[0].scatter(lon_all[::step], lat_all[::step], s=1, alpha=0.15, color=C1, linewidths=0)
    axes[0].set_title("All GPS fixes" if step == 1 else f"GPS fixes (1 in {step})")

    starts = np.array([[c[0][0], c[0][1]] for c in coords if c and c[0][0] is not None])
    ends   = np.array([[c[-1][0], c[-1][1]] for c in coords if c and c[-1][0] is not None])
    axes[1].scatter(starts[:, 0], starts[:, 1], s=7, alpha=0.6, color=C1, linewidths=0, label="origin")
    axes[1].scatter(ends[:, 0],   ends[:, 1],   s=7, alpha=0.6, color=C2, linewidths=0, label="destination")
    axes[1].set_title("Trip origins and destinations")
    axes[1].legend(loc="upper right", fontsize=8, labelcolor=INK)

    hb = axes[2].hexbin(lon_all, lat_all, gridsize=70, bins="log", cmap="Blues", mincnt=1, linewidths=0)
    axes[2].set_title("Fix density (log scale)")
    cb = fig.colorbar(hb, ax=axes[2], shrink=0.85); cb.set_label("fixes", color=MUTED, fontsize=8)
    cb.ax.tick_params(labelsize=7, colors=MUTED)

    for ax in axes:
        ax.set_aspect(aspect); ax.set_xlabel("lon"); ax.set_ylabel("lat")
        ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    fig.suptitle(f"{CITY} — spatial coverage", fontsize=13, fontweight="bold", color=INK, y=1.02)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{CITY}_spatial.png", bbox_inches="tight")
    plt.show()

## 7. Segments (`OSMids`)

The `OSMids` column is the route label — the thing a route-aware model actually
consumes. Two questions matter: what an id looks like (the two shipped cities use
different id schemes), and how the traversal counts are distributed over segments,
because a segment seen once cannot be learned.

In [ ]:
flat = [s for o in osmids for s in o]
seg_counter_fixes = Counter(flat)                                  # per GPS fix
seg_counter_trav  = Counter(s for o in osmids for s in runs(o))    # per traversal
seg_trips = Counter(s for o in osmids for s in set(o))             # distinct trips

n_seg = len(seg_counter_fixes)
suffixed = [s for s in seg_counter_fixes if "_" in s]
print(f"{n_seg:,} distinct segments over {len(flat):,} fixes / {sum(seg_counter_trav.values()):,} traversals")
print(f"ids carrying a '_<k>' split suffix : {pct(len(suffixed), n_seg)}")
print(f"purely numeric ids                 : {pct(sum(s.isdigit() for s in seg_counter_fixes), n_seg)}")
print(f"id length  min {min(map(len, seg_counter_fixes))}  max {max(map(len, seg_counter_fixes))}")
print("examples:", list(seg_counter_fixes)[:5])
if suffixed:
    base = {s.split("_")[0] for s in seg_counter_fixes}
    print(f"distinct base way ids              : {len(base):,} "
          f"({n_seg / len(base):.2f} segments per way on average)")
print()
c = np.array(sorted(seg_counter_trav.values(), reverse=True))
print(f"segments traversed once only : {pct(int((c == 1).sum()), n_seg)}")
print(f"segments in >= 10 trips      : {pct(int(sum(v >= 10 for v in seg_trips.values())), n_seg)}")
print(f"top 1% of segments carry     : {100 * c[:max(1, n_seg // 100)].sum() / c.sum():.1f}% of all traversals")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))

axes[0].loglog(np.arange(1, len(c) + 1), c, color=C1)
axes[0].set_title("Segment traversals, rank-frequency")
axes[0].set_xlabel("segment rank (log)"); axes[0].set_ylabel("traversals (log)")

axes[1].hist(np.log10(c), bins=40, color=C1, edgecolor="white", linewidth=0.4)
axes[1].set_title("Traversals per segment")
axes[1].set_xlabel("log10(traversals)"); axes[1].set_ylabel("segments")

rep = trips["repeat_osmid_frac"].dropna()
axes[2].hist(rep, bins=40, color=C3, edgecolor="white", linewidth=0.4)
axes[2].set_title("Consecutive fixes on the same segment")
axes[2].set_xlabel("fraction of a trip's fixes"); axes[2].set_ylabel("trips")

for ax in axes:
    ax.set_axisbelow(True)
fig.suptitle(f"{CITY} — segment usage", fontsize=13, fontweight="bold", color=INK, y=1.04)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{CITY}_segments.png", bbox_inches="tight")
plt.show()

top = pd.DataFrame(seg_counter_trav.most_common(10), columns=["osmid", "traversals"])
top["trips"] = top["osmid"].map(seg_trips)
print(top.to_string(index=False))

## 8. The directed edge list

`edge_list_directed_<city>.csv` is the graph a GNN would be built on. The check that
matters is **closure**: every segment the trips walk over should be a node in the graph,
and every consecutive pair of distinct segments should be an edge. Where that fails,
either the edge list was generated from a different network version — or, as with the
shipped Harbin file, it is cropped.

In [ ]:
if EDGES_PATH is None or not EDGES_PATH.exists():
    edges = None
    print("no edge list for this city — graph section skipped")
else:
    edges = pd.read_csv(EDGES_PATH, dtype=str)
    print(f"{len(edges):,} rows, columns {list(edges.columns)}")
    assert list(edges.columns) == ["osmid_u", "osmid_v"], "edge list does not match the contract"

    u, v = edges["osmid_u"], edges["osmid_v"]
    nodes = pd.unique(pd.concat([u, v], ignore_index=True))
    self_loops = int((u == v).sum())
    dup_rows   = int(edges.duplicated().sum())
    pairs      = set(zip(u, v))
    recip      = sum((b, a) in pairs for a, b in pairs if a != b)

    print(f"nodes (distinct segments)      : {len(nodes):,}")
    print(f"self loops (u == v)            : {pct(self_loops, len(edges))}")
    print(f"duplicate rows                 : {pct(dup_rows, len(edges))}")
    print(f"edges with the reverse present : {pct(recip, len(pairs))}")
    print(f"mean out-degree                : {len(edges) / len(nodes):.2f}")

In [ ]:
if edges is not None:
    no_loop = edges[edges.osmid_u != edges.osmid_v]
    outdeg = no_loop.groupby("osmid_u").size()
    indeg  = no_loop.groupby("osmid_v").size()
    deg = pd.DataFrame({"out": outdeg, "in": indeg}).reindex(nodes).fillna(0)

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
    bins = np.arange(0, min(int(deg[["in", "out"]].max().max()), 15) + 2) - 0.5
    axes[0].hist(deg["out"], bins=bins, color=C1, edgecolor="white", linewidth=0.4)
    axes[0].set_title("Out-degree (self loops removed)")
    axes[0].set_xlabel("out-degree"); axes[0].set_ylabel("segments")

    axes[1].hist(deg["in"], bins=bins, color=C1, edgecolor="white", linewidth=0.4)
    axes[1].set_title("In-degree (self loops removed)")
    axes[1].set_xlabel("in-degree"); axes[1].set_ylabel("segments")

    # how much of the graph the cropped trip sample actually touches
    seen = set(seg_counter_fixes)
    node_set = set(nodes)
    parts = [len(seen & node_set), len(seen - node_set), len(node_set - seen)]
    labels = ["in both", "trips only\n(not in graph)", "graph only\n(never driven)"]
    axes[2].bar(labels, parts, color=[C3, C2, GRID], edgecolor="white", linewidth=0.4)
    for x, val in enumerate(parts):
        axes[2].annotate(f"{val:,}", (x, val), ha="center", va="bottom", fontsize=8, color=INK)
    axes[2].set_title("Segments: trips vs edge list"); axes[2].set_ylabel("segments")

    for ax in axes:
        ax.set_axisbelow(True)
    fig.suptitle(f"{CITY} — road graph", fontsize=13, fontweight="bold", color=INK, y=1.04)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{CITY}_graph.png", bbox_inches="tight")
    plt.show()

    print(deg[["in", "out"]].describe(percentiles=[0.5, 0.95, 0.99]).round(2).to_string())

In [ ]:
if edges is not None:
    # Closure check 1 — every segment the trips visit should be a node in the graph.
    seen = set(seg_counter_fixes)
    missing_nodes = seen - node_set
    cov_nodes = 1 - len(missing_nodes) / len(seen)

    # Closure check 2 — every observed transition should be an edge in the graph.
    trans = Counter()
    for o in osmids:
        seq = runs(o)
        trans.update(zip(seq, seq[1:]))
    missing_trans = {k: n for k, n in trans.items() if k not in pairs}
    cov_trans_types = 1 - len(missing_trans) / len(trans)
    cov_trans_mass  = 1 - sum(missing_trans.values()) / sum(trans.values())

    print(f"segments visited by trips        : {len(seen):,}")
    print(f"  present in the edge list       : {cov_nodes:.2%}"
          + ("" if not missing_nodes else f"   ({len(missing_nodes):,} missing)"))
    print(f"distinct transitions in trips    : {len(trans):,}")
    print(f"  present as an edge             : {cov_trans_types:.2%} of transition types, "
          f"{cov_trans_mass:.2%} of traversals")
    if missing_trans:
        print("  most frequent missing transitions:")
        for (a, b), cnt in Counter(missing_trans).most_common(5):
            print(f"    {a} -> {b}   x{cnt}")
        print("  a low coverage here means the edge list and the trips do not come from the\n"
              "  same network snapshot, or the edge list file is truncated.")
    print(f"graph segments never driven in this sample: {pct(len(node_set - seen), len(node_set))}")

## 9. Road network geojson (when published)

The third file of the contract. It is the only place segment *geometry* and
`highway` class live; without it, map crops and length-based features cannot be built.

In [ ]:
if GEOJSON_PATH is None or not GEOJSON_PATH.exists():
    net = None
    print(f"no road network geojson for {CITY} — section skipped"
          + (f"\n({GEOJSON_PATH} not found)" if GEOJSON_PATH is not None else ""))
else:
    with open(GEOJSON_PATH, encoding="utf-8") as fh:
        gj = json.load(fh)
    feats = gj["features"]
    net = pd.DataFrame([f["properties"] for f in feats])
    net["n_points"] = [len(f["geometry"]["coordinates"]) for f in feats]
    print(f"{len(net):,} features, columns {list(net.columns)}")
    print(net.dtypes.to_string())
    expected_props = {"osmid", "original_osmid", "is_duplicate", "duplicate_index", "length", "highway"}
    print("contract properties present:", expected_props.issubset(set(net.columns)))
    if "osmid" in net:
        net["osmid"] = net["osmid"].astype(str)
        print("duplicate osmid rows:", int(net["osmid"].duplicated().sum()))
        print("segments used by trips but absent from the network:",
              f"{len(set(seg_counter_fixes) - set(net['osmid'])):,}")
    if "length" in net:
        print(net["length"].describe().round(2).to_string())
    if "highway" in net:
        print(net["highway"].astype(str).value_counts().head(12).to_string())

In [ ]:
if net is not None and {"length", "highway"}.issubset(net.columns):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    hist(axes[0], net["length"], log=True, title="Segment length", xlabel="metres (log)")
    axes[0].set_ylabel("segments")
    top_hw = net["highway"].astype(str).value_counts().head(10)[::-1]
    axes[1].barh(top_hw.index, top_hw.values, color=C1, edgecolor="white", linewidth=0.4)
    axes[1].set_title("Segments by highway class"); axes[1].set_xlabel("segments")
    for ax in axes:
        ax.set_axisbelow(True)
    fig.suptitle(f"{CITY} — road network", fontsize=13, fontweight="bold", color=INK, y=1.04)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{CITY}_network.png", bbox_inches="tight")
    plt.show()

## 10. Data-quality report

Every invariant in one table, plus the soft signals that decide whether a city is
usable as-is. `FAIL` breaks the contract; `WARN` is something to know before training
on it.

In [ ]:
checks = []
def check(name, ok, value, warn=False):
    checks.append({"check": name, "value": value,
                   "status": "ok" if ok else ("WARN" if warn else "FAIL")})

n = len(trips)
check("columns match the gold contract", list(raw.columns) == EXPECTED, str(list(raw.columns)))
check("all rows literal_eval",           len(bad_parse) == 0, f"{len(bad_parse)} failures")
check("list lengths agree",              bool(len_ok.all()), pct(int(len_ok.sum()), n))
check("Total_time == timestamp span",    bool(total_ok.all()), pct(int(total_ok.sum()), n))
check("timestamps non-decreasing",       bool(mono_ok.all()), pct(int(mono_ok.sum()), n))
check("timestamps strictly increasing",  bool(strict.all()), pct(int(strict.sum()), n), warn=True)
check("Id unique",                       int(raw['Id'].duplicated().sum()) == 0,
      f"{int(raw['Id'].duplicated().sum())} duplicates")
check("coordinates present",             has_coords, str(has_coords), warn=True)
check("no null cells",                   not bool(raw.isna().any().any()), str(raw.isna().sum().sum()))

check("trips >= 60 s",       short == 0, pct(short, n), warn=True)
check("trips >= 3 traversals", tiny == 0, pct(tiny, n), warn=True)
if trips["mean_speed_kmh"].notna().any():
    check("mean speed <= 120 km/h", fast == 0, pct(fast, n), warn=True)
    check("mean speed >= 2 km/h",   still == 0, pct(still, n), warn=True)
check("duplicated consecutive fixes",
      float(trips["dup_coord_frac"].fillna(0).mean()) < 0.05,
      f"{trips['dup_coord_frac'].fillna(0).mean():.1%} of fixes on average", warn=True)
check("trip starts span > 1 day", span_starts.days >= 1,
      f"{span_starts.days} d {span_starts.seconds // 3600} h", warn=True)
check("all 24 hours represented", trips["hour"].nunique() == 24,
      f"{trips['hour'].nunique()}/24 start hours", warn=True)

if edges is not None:
    check("edge list columns", list(edges.columns) == ["osmid_u", "osmid_v"], str(list(edges.columns)))
    check("no duplicate edges", dup_rows == 0, pct(dup_rows, len(edges)), warn=True)
    check("no self loops", self_loops == 0, pct(self_loops, len(edges)), warn=True)
    check("trip segments covered by the graph", cov_nodes > 0.999, f"{cov_nodes:.2%}")
    check("trip transitions covered by the graph", cov_trans_mass > 0.999,
          f"{cov_trans_mass:.2%} of traversals", warn=cov_trans_mass > 0.5)
else:
    check("edge list present", False, "missing")
check("road network geojson present", net is not None,
      "present" if net is not None else "missing", warn=True)

report = pd.DataFrame(checks)
bad = report[report.status != "ok"]
print(f"{CITY}: {len(report) - len(bad)} / {len(report)} checks clean")
report

In [ ]:
# One machine-readable summary per city, so section 11 can line them up.
summary = {
    "city": CITY,
    "note": NOTE,
    "trips_file": TRIPS_PATH.name,
    "n_trips": int(len(trips)),
    "n_fixes": int(trips["n_fixes"].sum()),
    "n_segments_seen": int(len(seg_counter_fixes)),
    "median_fixes": float(trips["n_fixes"].median()),
    "median_traversals": float(trips["n_traversals"].median()),
    "median_total_time_s": float(trips["total_time_s"].median()),
    "median_path_km": None if trips["path_km"].isna().all() else float(trips["path_km"].median()),
    "median_speed_kmh": None if trips["mean_speed_kmh"].isna().all() else float(trips["mean_speed_kmh"].median()),
    "median_gap_s": float(trips["gap_median_s"].median()),
    "zero_gap_frac": float((gaps == 0).mean()),
    "dup_coord_frac": float(trips["dup_coord_frac"].fillna(0).mean()),
    "repeat_osmid_frac": float(trips["repeat_osmid_frac"].fillna(0).mean()),
    "period_start": str(t0), "period_last_start": str(t1), "period_end": str(t_end),
    "days_covered": int(trips["date"].nunique()),
    "id_style": "uuid" if not str(raw["Id"].iloc[0]).isdigit() else "integer",
    "osmid_style": "way_split" if any("_" in s for s in seg_counter_fixes) else "numeric",
    "n_edges": None if edges is None else int(len(edges)),
    "n_graph_nodes": None if edges is None else int(len(nodes)),
    "self_loop_frac": None if edges is None else float(self_loops / len(edges)),
    "node_coverage": None if edges is None else float(cov_nodes),
    "transition_coverage": None if edges is None else float(cov_trans_mass),
    "has_geojson": net is not None,
    "checks_failed": int((report.status == "FAIL").sum()),
    "checks_warned": int((report.status == "WARN").sum()),
}
with open(OUT_DIR / f"eda_summary_{CITY}.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))

## 11. Side by side

Run the notebook once per city (change `CITY` in the first cell); this cell reads every
summary written so far and lines them up.

In [ ]:
files = sorted(OUT_DIR.glob("eda_summary_*.json"))
if len(files) < 2:
    print(f"only {len(files)} summary file(s) in {OUT_DIR}/ — "
          "re-run the notebook with a different CITY to fill this table")
comp = pd.DataFrame([json.load(open(f, encoding="utf-8")) for f in files]).set_index("city").T
comp

## What the two shipped cities look like

Numbers from a run over `datasets.zip` — both trip files cropped to the first 1000
trips — so the next person does not have to re-derive them:

| | **omsk** | **harbin** |
|---|---|---|
| `Id` | uuid string, **6 duplicated ids** in the crop | integer, unique |
| `OSMids` | plain numeric OSM ids (15–18 digits) | `<way>_<k>` split ids, 74% carry a suffix (2 979 base ways → 8 780 segments) |
| fixes per trip (median) | 134 | 39 |
| traversals per trip (median) | 55 | 29 |
| sampling interval (median) | **1 s**, and 45% of steps are 0 s | **30 s** |
| duplicated consecutive fixes | 45% of steps | none |
| `Total_time` (median) | 414 s | 1 334 s |
| path length (median) | 3.0 km | 9.9 km |
| mean speed (median) | 26.5 km/h | 26.4 km/h |
| period | 6–13 Dec 2020, 8 days, all 24 hours | every trip **starts within 28 s** of 2015-01-03 00:00 UTC |
| bounding box | ~28 x 37 km | ~21 x 19 km |
| edge list | 101 290 rows / 68 343 nodes, **24 559 self loops** (24%) | 62 030 rows / 23 102 nodes, no self loops |
| observed transitions present in the edge list | 100% | **21%** |
| implausible trips (>120 km/h or <2 km/h) | 0 | 23 (2.3%) |

Four things to carry forward:

* **The two cities are not sampled the same way.** Omsk is a ~1 Hz feed in which the
  same fix is repeated once per edge the matcher walks through — hence the 0-second
  gaps, the duplicated coordinates, the 60% repeat rate in `OSMids` and the self loops
  in the edge list. Harbin is a 30-second taxi feed with no repeats. Anything counted
  per GPS fix (sequence length, per-fix loss weights) measures the cadence rather than
  the city; collapse to traversals with `runs()` before comparing the two.
* **Harbin's edge list is truncated** — that is the crop, not a format break — so only
  21% of the transitions its trips actually make appear as edges. The same check on
  Omsk is 100%, which is what a complete pair of files looks like. Any graph work on
  Harbin needs the full file.
* **The Harbin crop is not a sample of the city.** All 1000 trips start in the same
  half-minute, so there is no time-of-day or day-of-week variation in it at all
  (the trips themselves run on to 03:37 UTC). Omsk's 1000 trips do span 8 days and all
  24 hours.
* **Neither city ships `road_network_unique_osmids_<city>.geojson`**, so segment
  geometry, length and `highway` class are unavailable here — section 9 stays empty
  until those files are added. Harbin also carries a tail of map-matching escapees
  (p99.9 path length 613 km, 451 km/h); filter on `mean_speed_kmh` before training.